# 03 - Data Cleaning and Preprocessing

**Purpose:** apply the approved cleaning policy from Notebook 02, save a cleaned analytical dataset, and produce audit metadata that supports EDA, modelling, and governance.

This notebook owns **cleaning execution and post-clean validation only**. It does not perform broad EDA, statistical testing, final encoding, scaling, resampling, or modelling.

## Stage ownership

| Stage | This notebook owns | Moved elsewhere |
|---|---|---|
| Apply cleaning function | Yes | Raw schema review belongs to Notebook 01 |
| Save cleaned dataset | Yes | Data-quality discovery belongs to Notebook 02 |
| Cleaning audit and flags | Yes | Portfolio EDA belongs to Notebook 04 |
| Feature policy | Yes | Feature engineering finalization belongs to Notebook 05 |
| Column lineage | Yes | Encoding/scaling/resampling belongs to Notebook 06 |
| Post-clean validation | Yes | Statistical tests belong to Notebook 04 |

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
TABLE_DIR = REPORT_DIR / "tables"
FIGURE_DIR = REPORT_DIR / "figures"

for path in [INTERIM_DIR, PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TARGET_COL = "defaulter"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


from credit_risk.data.cleaning import clean_credit_risk_dataset

INTERIM_PATH = INTERIM_DIR / "credit_risk_merged_interim.csv"
PROCESSED_CSV_PATH = PROCESSED_DIR / "credit_risk_cleaned.csv"
PROCESSED_PARQUET_PATH = PROCESSED_DIR / "credit_risk_cleaned.parquet"

if not INTERIM_PATH.exists():
    raise FileNotFoundError(
        f"Interim dataset not found at {INTERIM_PATH}. Run Notebook 01 or scripts/run_data_pipeline.py first."
    )

raw_df = pd.read_csv(INTERIM_PATH, low_memory=False)
raw_df.shape

(134417, 25)

## 1. Load cleaning decisions from Notebook 02

In [2]:
decision_log_path = TABLE_DIR / "02_data_quality_decision_log_for_notebook_03.csv"

if decision_log_path.exists():
    cleaning_decision_log = pd.read_csv(decision_log_path)
else:
    cleaning_decision_log = pd.DataFrame([
        {"area": "Observation grain", "decision": "Preserve user_id + record_sequence.", "next_stage": "Notebook 03"},
        {"area": "Missing amount", "decision": "Create amount_missing_flag and retain rows.", "next_stage": "Notebook 03 / 06"},
        {"area": "Leakage fields", "decision": "Exclude repayment-derived variables from baseline modelling.", "next_stage": "Notebook 05 / 06"},
    ])

cleaning_decision_log.to_csv(TABLE_DIR / "03_cleaning_policy_from_quality_review.csv", index=False)
cleaning_decision_log

,area,decision,next_stage
0,Observation grain,Preserve user_id + record_sequence.,Notebook 03 cleaning
1,Missing amount,Do not drop rows; create amount_missing_flag and impute later in modelling pipeline.,Notebook 03 / 06
2,Invalid amount,Flag non-positive amount and convert to missing in cleaning.,Notebook 03
3,Employment missingness,Create flags and fill categorical missing values as Unknown.,Notebook 03
4,Placeholder zero,Treat placeholder 0 in industry/work_experience as Unknown/unavailable.,Notebook 03
5,Leakage fields,Exclude repayment-derived fields from baseline model; retain for monitoring.,Notebook 03 / 04 / 05
6,Sensitive/proxy variables,Retain for governance; exclude high-risk fields from baseline model.,Notebook 03 / 05
7,Preprocessing leakage,"Do not fit encoders, scalers, resamplers, or transformations before train/test split.",Notebook 06


## 2. Apply centralized cleaning function

In [3]:
cleaning_result = clean_credit_risk_dataset(raw_df)

cleaned_df = cleaning_result.cleaned
audit_summary = cleaning_result.audit_summary
flag_summary = cleaning_result.flag_summary
model_feature_policy = cleaning_result.model_feature_policy

required_outputs = {
    "cleaned_df": cleaned_df,
    "audit_summary": audit_summary,
    "flag_summary": flag_summary,
    "model_feature_policy": model_feature_policy,
}

for name, obj in required_outputs.items():
    if obj is None:
        raise ValueError(f"{name} was not created by clean_credit_risk_dataset().")

display(cleaned_df.head())

,user_id,loan_category,amount,interest_rate,tenure_years,record_sequence,employment_type,tier_of_employment,industry,role,work_experience,total_income_pa,gender,married,dependents,home,pincode,social_profile,is_verified,delinq_2yrs,total_payment,received_principal,interest_received,number_of_loans,defaulter,amount_missing_raw_flag,amount_non_positive_flag,industry_placeholder_zero_flag,work_experience_placeholder_zero_flag,amount_missing_flag,employment_type_missing_flag,tier_of_employment_missing_flag,industry_missing_flag,work_experience_missing_flag,married_missing_flag,social_profile_missing_flag,is_verified_missing_flag,loan_to_income_ratio,payment_to_amount_ratio,principal_to_amount_ratio,interest_to_amount_ratio,principal_exceeds_amount_flag,core_data_quality_issue_count,has_core_data_quality_issue,broad_data_quality_issue_count,has_broad_data_quality_issue
0,7013527,Consolidation,"55,884.0000",11.8400,6,1,Salaried,B,mLVIVxoGY7TUDJ1FyFoSIZi1SFcaBmO01AydRchaEiGYtUhXGgZjtNQMnUXIWhIhmLVIVxoGY7TUDJ1FyFoSIUGp9a+9oBSLvyI5Jdz9fNg=,KHMbckjadbckIFGAZSEWdkcndwkcnCCM,1-2,"125,000.0000",Female,Yes,4,Rent,XX852X,No,Unknown,0,"1,824.1500",971.4600,852.6900,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0.4471,0.0326,0.0174,0.0153,0,0,0,0,0
1,7014291,Consolidation,"55,511.0000",16.9400,4,1,Self-Employed,D,mLVIVxoGY7TUDJ1FyFoSIZi1SFcaBmO01AydRchaEiGYtUhXGgZjtNQMnUXIWhIhmLVIVxoGY7TUDJ1FyFoSIasttBX92VTWzdldgDGNrI4=,KHMbckjadbckIFGNCSEWdkcndwkcnCCM,10+,"61,000.0000",Female,No,1,Mortgage,XX286X,Unknown,Source Verified,0,"22,912.5330","18,000.0000","4,912.5300",0,0,0,0,0,0,0,0,0,0,0,0,1,0,0.9100,0.4128,0.3243,0.0885,0,0,0,0,0
2,7014327,Consolidation,"12,289.0000",11.8400,6,1,Unknown,Unknown,mLVIVxoGY7TUDJ1FyFoSIZi1SFcaBmO01AydRchaEiGYtUhXGgZjtNQMnUXIWhIhYq8O0ygAy1wZSaxqrNLNYfOibgdA1QeQhNPkLJcwKVc=,KHMbckjadbckIFGNYSEWdkcndwkcnCCM,5-10,"100,000.0000",Other,Unknown,3,Own,XX113X,No,Unknown,0,"7,800.4400","4,489.7600","3,310.6800",0,1,0,0,0,0,0,1,1,0,0,1,0,1,0.1229,0.6347,0.3653,0.2694,0,0,0,0,0
3,7014304,Credit Card,"29,324.0000",14.7100,4,1,Unknown,Unknown,mLVIVxoGY7TUDJ1FyFoSIZi1SFcaBmO01AydRchaEiGYtUhXGgZjtNQMnUXIWhIhU4gDxq3nasE2lBvkF3BByiHO4FJWtkam4oglyfSc5To=,KHMbckjadbckIFGCASEWdkcndwkcnCCM,2-3,"30,000.0000",Male,Unknown,1,Rent,XX941X,Yes,Unknown,0,"6,672.0500","5,212.2900","1,459.7600",0,0,0,0,0,0,0,1,1,0,0,1,0,1,0.9775,0.2275,0.1777,0.0498,0,0,0,0,0
4,7031995,Credit Card,"30,252.0000",14.7100,4,1,Unknown,Unknown,mLVIVxoGY7TUDJ1FyFoSIZi1SFcaBmO01AydRchaEiGYtUhXGgZjtNQMnUXIWhIhd9MxBQLjUjv++jfpgYoMqnBasBZFnH48Gq5+cHbJ+zg=,KHMbckjadbckIFGCASEWdkcndwkcnCCM,10+,"65,000.0000",Male,Unknown,3,Rent,XX913X,No,Verified,0,"11,793.0013","10,000.0000","1,793.0000",0,0,0,0,0,0,0,1,1,0,0,1,0,0,0.4654,0.3898,0.3306,0.0593,0,0,0,0,0


## 3. Save cleaned dataset and core audit outputs

In [4]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

cleaned_df.to_csv(PROCESSED_CSV_PATH, index=False)

parquet_status = "not_attempted"
try:
    cleaned_df.to_parquet(PROCESSED_PARQUET_PATH, index=False)
    parquet_status = "saved"
except ImportError:
    parquet_status = "skipped_pyarrow_or_fastparquet_missing"

audit_summary.to_csv(TABLE_DIR / "03_cleaning_audit_summary.csv", index=False)
flag_summary.to_csv(TABLE_DIR / "03_cleaning_flag_summary.csv", index=False)
model_feature_policy.to_csv(TABLE_DIR / "03_model_feature_policy.csv", index=False)

cleaned_dataset_save_summary = pd.DataFrame({
    "output": [
        "cleaned_csv",
        "cleaned_parquet",
        "cleaning_audit_summary",
        "cleaning_flag_summary",
        "model_feature_policy",
    ],
    "path": [
        str(PROCESSED_CSV_PATH.relative_to(PROJECT_ROOT)),
        str(PROCESSED_PARQUET_PATH.relative_to(PROJECT_ROOT)),
        str((TABLE_DIR / "03_cleaning_audit_summary.csv").relative_to(PROJECT_ROOT)),
        str((TABLE_DIR / "03_cleaning_flag_summary.csv").relative_to(PROJECT_ROOT)),
        str((TABLE_DIR / "03_model_feature_policy.csv").relative_to(PROJECT_ROOT)),
    ],
    "status": ["saved", parquet_status, "saved", "saved", "saved"],
})

cleaned_dataset_save_summary.to_csv(TABLE_DIR / "03_cleaned_dataset_save_summary.csv", index=False)
cleaned_dataset_save_summary

,output,path,status
0,cleaned_csv,data\processed\credit_risk_cleaned.csv,saved
1,cleaned_parquet,data\processed\credit_risk_cleaned.parquet,saved
2,cleaning_audit_summary,reports\tables\03_cleaning_audit_summary.csv,saved
3,cleaning_flag_summary,reports\tables\03_cleaning_flag_summary.csv,saved
4,model_feature_policy,reports\tables\03_model_feature_policy.csv,saved


## 4. Post-clean validation checks

In [5]:
raw_columns = set(raw_df.columns)
cleaned_columns = set(cleaned_df.columns)

columns_removed = sorted(raw_columns - cleaned_columns)
columns_added = sorted(cleaned_columns - raw_columns)
columns_kept = sorted(raw_columns & cleaned_columns)

validation_checks = pd.DataFrame({
    "check": [
        "row_count_preserved",
        "target_present",
        "record_sequence_present",
        "record_key_duplicate_count",
        "cleaned_column_count",
        "columns_added_count",
        "columns_removed_count",
        "remaining_missing_value_count",
    ],
    "result": [
        raw_df.shape[0] == cleaned_df.shape[0],
        TARGET_COL in cleaned_df.columns,
        "record_sequence" in cleaned_df.columns,
        int(cleaned_df[["user_id", "record_sequence"]].duplicated().sum())
        if {"user_id", "record_sequence"}.issubset(cleaned_df.columns)
        else np.nan,
        cleaned_df.shape[1],
        len(columns_added),
        len(columns_removed),
        int(cleaned_df.isna().sum().sum()),
    ],
})

column_lineage = pd.concat(
    [
        pd.DataFrame({"column": columns_kept, "lineage_status": "kept_from_raw"}),
        pd.DataFrame({"column": columns_added, "lineage_status": "added_during_cleaning"}),
        pd.DataFrame({"column": columns_removed, "lineage_status": "removed_during_cleaning"}),
    ],
    ignore_index=True,
)

validation_checks.to_csv(TABLE_DIR / "03_cleaning_validation_checks.csv", index=False)
column_lineage.to_csv(TABLE_DIR / "03_column_lineage.csv", index=False)

display(validation_checks)
display(column_lineage)

,check,result
0,row_count_preserved,True
1,target_present,True
2,record_sequence_present,True
3,record_key_duplicate_count,0
4,cleaned_column_count,46
5,columns_added_count,21
6,columns_removed_count,0
7,remaining_missing_value_count,138190


,column,lineage_status
0,amount,kept_from_raw
1,defaulter,kept_from_raw
2,delinq_2yrs,kept_from_raw
3,dependents,kept_from_raw
4,employment_type,kept_from_raw
5,gender,kept_from_raw
6,home,kept_from_raw
7,industry,kept_from_raw
8,interest_rate,kept_from_raw
9,interest_received,kept_from_raw


## 5. Post-clean target and missingness validation

In [6]:
post_clean_target_distribution = (
    cleaned_df[TARGET_COL]
    .value_counts(dropna=False)
    .rename_axis(TARGET_COL)
    .reset_index(name="row_count")
    .sort_values(TARGET_COL, na_position="last")
)
post_clean_target_distribution["share_pct"] = (
    post_clean_target_distribution["row_count"] / len(cleaned_df) * 100
).round(4)

post_clean_missingness = pd.DataFrame({
    "column": cleaned_df.columns,
    "dtype": cleaned_df.dtypes.astype(str).values,
    "missing_count": cleaned_df.isna().sum().values,
    "missing_pct": (cleaned_df.isna().mean().values * 100).round(2),
    "unique_values": cleaned_df.nunique(dropna=True).values,
})
post_clean_missingness = post_clean_missingness.query("missing_count > 0").sort_values(
    ["missing_pct", "missing_count"], ascending=False
)

post_clean_target_distribution.to_csv(TABLE_DIR / "03_post_clean_target_distribution.csv", index=False)
post_clean_missingness.to_csv(TABLE_DIR / "03_post_clean_missingness.csv", index=False)

display(post_clean_target_distribution)
display(post_clean_missingness)

,defaulter,row_count,share_pct
0,0,122264,90.9587
1,1,12153,9.0413


,column,dtype,missing_count,missing_pct,unique_values
2,amount,float64,27638,20.5600,86156
37,loan_to_income_ratio,float64,27638,20.5600,104531
38,payment_to_amount_ratio,float64,27638,20.5600,106764
39,principal_to_amount_ratio,float64,27638,20.5600,106248
40,interest_to_amount_ratio,float64,27638,20.5600,106695


## 6. Quality flag dictionary

In [7]:
flag_cols = [col for col in cleaned_df.columns if col.endswith("_flag") or col.startswith("has_")]
count_cols = [col for col in cleaned_df.columns if col.endswith("_issue_count") or col.endswith("_missing_count")]

quality_flag_dictionary = pd.DataFrame({
    "field": flag_cols + count_cols,
    "field_type": ["binary_flag"] * len(flag_cols) + ["count_feature"] * len(count_cols),
})
quality_flag_dictionary["business_meaning"] = quality_flag_dictionary["field"].str.replace("_", " ", regex=False)
quality_flag_dictionary["recommended_use"] = np.where(
    quality_flag_dictionary["field"].str.contains("quality|missing|placeholder|non_positive|exceeds", case=False, na=False),
    "audit_monitoring_and_possible_model_feature_after_policy_review",
    "review_before_use",
)

quality_flag_summary = []
for col in flag_cols:
    if col in cleaned_df.columns and pd.api.types.is_numeric_dtype(cleaned_df[col]):
        quality_flag_summary.append({
            "flag": col,
            "flagged_count": int((cleaned_df[col] == 1).sum()),
            "flagged_pct": round((cleaned_df[col] == 1).mean() * 100, 4),
        })

quality_flag_summary = pd.DataFrame(quality_flag_summary).sort_values("flagged_pct", ascending=False) if quality_flag_summary else pd.DataFrame()

quality_flag_dictionary.to_csv(TABLE_DIR / "03_quality_flag_dictionary.csv", index=False)
quality_flag_summary.to_csv(TABLE_DIR / "03_quality_flag_summary_from_cleaned_data.csv", index=False)

display(quality_flag_dictionary)
display(quality_flag_summary)

,field,field_type,business_meaning,recommended_use
0,amount_missing_raw_flag,binary_flag,amount missing raw flag,audit_monitoring_and_possible_model_feature_after_policy_review
1,amount_non_positive_flag,binary_flag,amount non positive flag,audit_monitoring_and_possible_model_feature_after_policy_review
2,industry_placeholder_zero_flag,binary_flag,industry placeholder zero flag,audit_monitoring_and_possible_model_feature_after_policy_review
3,work_experience_placeholder_zero_flag,binary_flag,work experience placeholder zero flag,audit_monitoring_and_possible_model_feature_after_policy_review
4,amount_missing_flag,binary_flag,amount missing flag,audit_monitoring_and_possible_model_feature_after_policy_review
5,employment_type_missing_flag,binary_flag,employment type missing flag,audit_monitoring_and_possible_model_feature_after_policy_review
6,tier_of_employment_missing_flag,binary_flag,tier of employment missing flag,audit_monitoring_and_possible_model_feature_after_policy_review
7,industry_missing_flag,binary_flag,industry missing flag,audit_monitoring_and_possible_model_feature_after_policy_review
8,work_experience_missing_flag,binary_flag,work experience missing flag,audit_monitoring_and_possible_model_feature_after_policy_review
9,married_missing_flag,binary_flag,married missing flag,audit_monitoring_and_possible_model_feature_after_policy_review


,flag,flagged_count,flagged_pct
14,has_broad_data_quality_issue,116527,86.6907
7,industry_missing_flag,113583,84.5005
8,work_experience_missing_flag,113583,84.5005
2,industry_placeholder_zero_flag,113579,84.4975
3,work_experience_placeholder_zero_flag,113579,84.4975
5,employment_type_missing_flag,79686,59.2827
6,tier_of_employment_missing_flag,79686,59.2827
9,married_missing_flag,45084,33.5404
10,social_profile_missing_flag,44846,33.3633
11,is_verified_missing_flag,33507,24.9277


## 7. Feature policy outputs

In [8]:
display(Markdown("### Model feature policy"))
display(model_feature_policy)

feature_policy_path = TABLE_DIR / "03_model_feature_policy.csv"
if not model_feature_policy.empty:
    excluded_feature_list = model_feature_policy[
        model_feature_policy.astype(str).apply(lambda row: row.str.contains("exclude|leakage|target|sensitive|proxy", case=False, na=False).any(), axis=1)
    ].copy()
else:
    excluded_feature_list = pd.DataFrame()

monitoring_only_candidates = [
    "total_payment", "received_principal", "interest_received",
    "gender", "pincode", "social_profile", "user_id", "record_sequence",
]
monitoring_only_feature_list = pd.DataFrame({
    "feature": [col for col in monitoring_only_candidates if col in cleaned_df.columns],
    "recommended_use": "monitoring_or_governance_only_not_first_baseline_model",
})

excluded_feature_list.to_csv(TABLE_DIR / "03_excluded_feature_list.csv", index=False)
monitoring_only_feature_list.to_csv(TABLE_DIR / "03_monitoring_only_feature_list.csv", index=False)

display(Markdown("### Monitoring-only feature list"))
display(monitoring_only_feature_list)

### Model feature policy

,column,recommended_use,reason
0,user_id,exclude_from_model,Identifier; useful for joins and audit only.
1,record_sequence,exclude_from_model,Technical merge key; no business meaning.
2,defaulter,target_only,Target variable; never used as predictor.
3,total_payment,exclude_from_baseline_model,May include post-origination repayment behaviour and can create target leakage.
4,received_principal,exclude_from_baseline_model,May be observed after the lending decision or outcome window.
5,interest_received,exclude_from_baseline_model,May be post-outcome repayment information.
6,payment_to_amount_ratio,exclude_from_baseline_model,Derived from post-origination repayment information; useful for monitoring but leakage-prone for default prediction.
7,principal_to_amount_ratio,exclude_from_baseline_model,Derived from received principal; may reveal repayment outcome information.
8,interest_to_amount_ratio,exclude_from_baseline_model,Derived from interest received; may reveal repayment/outcome timing.
9,gender,fairness_audit_only,"Sensitive/proxy field; use for bias diagnostics, not model training."


### Monitoring-only feature list

,feature,recommended_use
0,total_payment,monitoring_or_governance_only_not_first_baseline_model
1,received_principal,monitoring_or_governance_only_not_first_baseline_model
2,interest_received,monitoring_or_governance_only_not_first_baseline_model
3,gender,monitoring_or_governance_only_not_first_baseline_model
4,pincode,monitoring_or_governance_only_not_first_baseline_model
5,social_profile,monitoring_or_governance_only_not_first_baseline_model
6,user_id,monitoring_or_governance_only_not_first_baseline_model
7,record_sequence,monitoring_or_governance_only_not_first_baseline_model


## 8. Encoding and transformation plan

In [9]:
encoding_and_transformation_plan = pd.DataFrame([
    {"stage": "Notebook 03", "operation": "missing flags", "status": "created where policy requires", "leakage_control": "safe before split because flags describe source missingness"},
    {"stage": "Notebook 03", "operation": "clean invalid amount", "status": "flag first, then convert invalid values to missing", "leakage_control": "rule-based cleaning only"},
    {"stage": "Notebook 03", "operation": "one-hot encoding", "status": "not performed", "leakage_control": "fit after train/test split inside modelling pipeline"},
    {"stage": "Notebook 03", "operation": "ordinal encoding", "status": "not performed", "leakage_control": "fit after train/test split inside modelling pipeline"},
    {"stage": "Notebook 03", "operation": "scaling", "status": "not performed", "leakage_control": "fit on training data only"},
    {"stage": "Notebook 03", "operation": "log transform / winsorization", "status": "not performed", "leakage_control": "fit/choose inside feature-engineering or modelling pipeline after split"},
    {"stage": "Notebook 03", "operation": "SMOTE or resampling", "status": "not performed", "leakage_control": "training data only"},
])

encoding_and_transformation_plan.to_csv(TABLE_DIR / "03_encoding_and_transformation_plan.csv", index=False)
encoding_and_transformation_plan

,stage,operation,status,leakage_control
0,Notebook 03,missing flags,created where policy requires,safe before split because flags describe source missingness
1,Notebook 03,clean invalid amount,"flag first, then convert invalid values to missing",rule-based cleaning only
2,Notebook 03,one-hot encoding,not performed,fit after train/test split inside modelling pipeline
3,Notebook 03,ordinal encoding,not performed,fit after train/test split inside modelling pipeline
4,Notebook 03,scaling,not performed,fit on training data only
5,Notebook 03,log transform / winsorization,not performed,fit/choose inside feature-engineering or modelling pipeline after split
6,Notebook 03,SMOTE or resampling,not performed,training data only


## Carry-forward decisions

1. Use the cleaned dataset saved at `data/processed/credit_risk_cleaned.csv`.
2. Treat `user_id + record_sequence` as the observation grain.
3. Keep leakage-risk, sensitive/proxy, and monitoring-only variables available for audit and monitoring, not for the first baseline model.
4. Do not encode, scale, transform, resample, or model in this notebook.
5. Notebook 04 should use the cleaned dataset for portfolio EDA and statistical analysis.